# Introduction to APIs

Application Programming Interfaces (APIs) are interfaces between computers or pieces of software. A given user or enterprise may continuously produce data, and creating an API is generally the procedure follow for other users to seamlessly access the data.

Check out [this video](https://share.vidyard.com/watch/Le2QkL3lixu3GmssEXYyLg) for a better view on how APIs work and what they are used for.

APIs provide great interfaces to access data from sites. Frequently, a user is given an access token which must be supplied on the query. Some APIs are free of charge, with some requiring registration for obtaining the access token, others are paid. Additional requirements to the desired data should also be supplied on the queried.

Some APIs have Python libraries, making the process of collecting data easier.

Some examples of how APIs can be used are shown below:

- **Weather API ([IPMA](https://api.ipma.pt/)):** weather dashboard, focused on reporting realtime weather, such as temperature, humidity, ondulation, UV risk, etc. The API should be queried frequently, for the dashboard to receive the most updated data.

- **Seismic Activity ([IPMA](https://api.ipma.pt/)):** seismic events in Portugal and surrounding regions. After the data is gathered using the API, it is plotted in an interactive map.

- [**iTunes Search API**](https://developer.apple.com/library/archive/documentation/AudioVideo/Conceptual/iTuneSearchAPI/index.html)**:** data science study of songs. The API is queried for songs by a certain artist, receiving for each song parameters such as the duration, release date, album and genre.


### For constructing your own API, check out [this ebook](https://www.mulesoft.com/lp/ebook/api/restbook).

# API Example 1 -- Minimum and Maximum Temperatures 

Based on data from IPMA regarding minimum and maximum temperatures for a given municipality, and for the past 3 months. The API requires 3 parameters: district, DICO code, and municipality. Examples are shown below.

https://api.ipma.pt/

In [ ]:
district = 'lisboa'; dico = '1106-lisboa'
#district = 'porto'; dico = '1312-porto'
#district = 'faro'; dico = '0801-albufeira'
#district = 'guarda'; dico = '0908-manteigas'


In [ ]:
## Correctly Format the URLs
min_file = 'https://api.ipma.pt/open-data/observation/climate/temperature-min/{}/mtnmn-{}.csv'.format(district, dico)
max_file = 'https://api.ipma.pt/open-data/observation/climate/temperature-max/{}/mtxmx-{}.csv'.format(district, dico)
print(min_file)
print(max_file)

In [ ]:
## Load the files into pandas dataframes
import pandas as pd
min_df = pd.read_csv(min_file)
max_df = pd.read_csv(max_file)
min_df.head()

### Using Matplotlib for plotting the temperatures

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot([datetime.strptime(i, '%Y-%m-%d') for i in min_df['date'].to_list()],
         min_df['mean'].to_list(), 'b-')
ax.plot([datetime.strptime(i, '%Y-%m-%d') for i in min_df['date'].to_list()],
         min_df['minimum'].to_list(), 'b--')
ax.plot([datetime.strptime(i, '%Y-%m-%d') for i in min_df['date'].to_list()],
         min_df['maximum'].to_list(), 'b--')

ax.plot([datetime.strptime(i, '%Y-%m-%d') for i in max_df['date'].to_list()],
         max_df['mean'].to_list(), 'r-')
ax.plot([datetime.strptime(i, '%Y-%m-%d') for i in max_df['date'].to_list()],
         max_df['minimum'].to_list(), 'r--')
ax.plot([datetime.strptime(i, '%Y-%m-%d') for i in max_df['date'].to_list()],
         max_df['maximum'].to_list(), 'r--')

ax.set_ylabel('Temperature (ºC)')
ax.set_title(district + '/' + dico)

# API Example 2 -- Seismic Activity

Based on data from IPMA regarding seismic activity

idArea = 7 -> continental Portugal and Madeira

idArea = 3 -> Azores

In [ ]:
idArea = 7

In [ ]:
import urllib.request, json
seismic_file = 'https://api.ipma.pt/open-data/observation/seismic/{}.json'.format(idArea)

with urllib.request.urlopen(seismic_file) as url:
    data = json.loads(url.read().decode())

print(seismic_file)

The data can be converted from its JSON format to a pandas dataframe

In [ ]:
import pandas as pd
df = pd.DataFrame(data['data'])

df['text'] = '<b>' + df['obsRegion'] + '</b><br>' + df['time'].str[:10] + ' ' + df['time'].str[11:]
df['text'] = df['text'] + '<br>Depth: ' + (df['depth']).astype(str)
df['text'] = df['text'] + 'm<br>Magnitude: ' + (df['magnitud']).astype(str)
df['text'] = df['text'] + '<br>(' + df['lon'] + ',' + df['lat'] + ')'

df['magnitud'] = df['magnitud'].astype(float)
# Drop values with negative magnitude
df = df[df['magnitud'] >= 0]

df['depth_inv'] = df['depth'].astype(float)
#df['depth_inv'] = -df['depth_inv'] + max(df['depth_inv'])

df['lat'] = df['lat'].astype(float)
df['lon'] = df['lon'].astype(float)
df['time'] = pd.to_datetime(df['time'], format='%Y-%m-%dT%H:%M:%S')
df['timevar'] = df['time'] - min(df['time'])
df['timevar'] = df['timevar'].dt.days + df['timevar'].dt.seconds/24/3600
df

### Using Plotly for plotting an interactive map

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"
import plotly.express as px

In [ ]:
fig = px.scatter_map(df, lat="lat", lon="lon", size="depth_inv", color="magnitud", text='text', size_max=15, zoom=3)
fig.update_traces(hovertemplate='%{text}')
fig.show()

# API Example 3 -- iTunes Songs Analysis

In [ ]:
import requests
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
artist_name = 'Coldplay'
limit_search = 50

url = 'https://itunes.apple.com/search'

params = {
    'term': artist_name,
    'entity': 'song',
    'limit': limit_search
}

response = requests.get(url, params=params)

print(response.status_code)

data = response.json()

print(data['resultCount'])

In [ ]:
songs = data['results']

songs_df = pd.DataFrame(songs)

songs_df.head()

In [ ]:
tracks_df = songs_df[
    ['artistName', 'trackName', 'collectionName',
     'releaseDate', 'trackTimeMillis',
     'primaryGenreName']
].copy()

tracks_df['releaseDate'] = pd.to_datetime(tracks_df['releaseDate'])

tracks_df['duration'] = tracks_df['trackTimeMillis'] / 60000

tracks_df.head()

In [ ]:
ax = sns.scatterplot(
    data=tracks_df,
    x='releaseDate',
    y='duration',
    hue='collectionName'
)

ax.legend(
    bbox_to_anchor=(1, 1),
    loc='upper left',
    title='Album'
)

ax.set_title(
    'Duration of ' + artist_name.title() + ' Songs Over Time'
)

ax.set_xlabel('Release Date')
ax.set_ylabel('Duration (minutes)')

In [ ]:
ax = sns.boxplot(
    data=tracks_df,
    x='collectionName',
    y='duration'
)

ax.set_title(
    'Song Duration of ' + artist_name.title() + ' by Album'
)

ax.set_xlabel('Album')
ax.set_ylabel('Duration (minutes)')

plt.xticks(rotation=45, ha='right')
plt.show()